# 📘 M2C2 DataKit Notebook: Universal Loading, Assurance, and Scoring

This notebook demonstrates a full analytic pipeline using the `m2c2-datakit` package. It uses the `LASSIE` class to load, validate, score, and optionally export data from multiple source types.

---

## 🎯 Purpose

Enable researchers to plug in data from varied sources (e.g., MongoDB, UAS, MetricWire, CSV bundles) and apply a consistent pipeline for:

- Input validation

- Scoring via predefined rules

- Inspection and summarization

- Tidy export and codebook generation

---

## Inspired by:

<img src="https://m.media-amazon.com/images/M/MV5BNDNkZDk0ODktYjc0My00MzY4LWE3NzgtNjU5NmMzZDA3YTA1XkEyXkFqcGc@._V1_FMjpg_UX1000_.jpg" alt="Inspiration for Package, Lassie Movie" width="100"/>

## 🧠 L.A.S.S.I.E. Pipeline Summary

| Step | Method           | Purpose                                                                 |
|------|------------------|-------------------------------------------------------------------------|
| L    | `load()`         | Load raw data from a supported source (e.g., MongoDB, UAS, MetricWire). |
| A    | `assure()`       | Validate that required columns exist before processing.                 |
| S    | `score()`        | Apply scoring logic based on predefined or custom rules.                |
| S    | `summarize()`    | Aggregate scored data by participant, session, or custom groups.        |
| I    | `inspect()`      | Visualize distributions or pairwise plots for quality checks.           |
| E    | `export()`       | Save scored and summarized data to tidy files and optionally metadata.  |

---


## 📦 Supported Sources

You may have used M2C2kit tasks via our various integrations, including the ones listed below. Each integration has its own loader class, which is responsible for reading the data and converting it into a format that can be processed by the `m2c2_datakit` package. Keep in mind that you are responsible for ensuring that the data is in the correct format for each loader class.

In the future we anticipate creating loaders for downloading data via API.

| Source Type   | Loader Class          | Key Arguments                            | Notes                                 |
|---------------|------------------------|-------------------------------------------|----------------------------------------|
| `mongodb`     | `MongoDBImporter`      | `source_path` (URL, to JSON)                      | Expects flat or nested JSON documents. |
| `multicsv`    | `MultiCSVImporter`     | `source_map` (dict of CSV paths)          | Each activity type is its own file.    |
| `metricwire`  | `MetricWireImporter`   | `source_path` (glob pattern or default)   | Processes JSON files from unzipped export. |
| `qualtrics`    | `QualtricsImporter`     | `source_path` (URL to CSV)         | Each activity's trial saves data to a new column.    |
| `uas`         | `UASImporter`          | `source_path` (URL, to pseudo-JSON)                       | Parses newline-delimited JSON.         |

## Verify Data

## 🛠️ Setup for Developers of this Package

## Setup Environment to run Notebook

This section is for developer use only. Ignore the two code chunks below if you are a user of the package..

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade m2c2_datakit

In [ ]:
!uv pip show m2c2_datakit


In [ ]:
import os
import re
import glob
import json
from dotenv import load_dotenv
import m2c2_datakit as m2c2
m2c2.core.get_package_version()

## Configure output folder and summary functions

In [4]:
output_folder = "tidy"

summary_func_map = {
    "Symbol Search": m2c2.tasks.symbol_search.summarize,
    "Grid Memory": m2c2.tasks.grid_memory.summarize,
    "Color Shapes": m2c2.tasks.color_shapes.summarize,
    
    "symbol-search": m2c2.tasks.symbol_search.summarize,
    "grid-memory": m2c2.tasks.grid_memory.summarize,
}

# Note: ^ this also means that a user could specify their own summarize functions as needed!

## Platform: Metricwire

In [ ]:
# Define source_folder relative to current working directory
source_folder_mw = os.path.abspath(os.path.join(os.pardir, 'datakit/data/metricwire/unzipped'))
source_path_mw = f"{source_folder_mw}/*/*/*.json"

# Data from Metricwire
mw = m2c2.core.pipeline.LASSIE().load(source_name="metricwire", source_path=source_path_mw)
mw.assure(required_columns=m2c2.core.config.settings.STANDARD_GROUPING_FOR_AGGREGATION_METRICWIRE)
mw.score()

In [ ]:
mw.summarize(summary_func_map = summary_func_map, groupby_cols=m2c2.core.config.settings.STANDARD_GROUPING_FOR_AGGREGATION_METRICWIRE)
mw.inspect()
mw.export(file_basename="export_metricwire", directory=output_folder)
mw.export_codebook(filename="codebook_metricwire.md", directory=output_folder)

## Platform: MongoDB

In [ ]:
# Define source_folder relative to current working directory
#source_folder_mdb = os.path.abspath(os.path.join(os.pardir, "datakit/data/production-mongo-export"))
#source_path_mdb = f"{source_folder_mdb}/data_exported_120424_1010am.json"

source_path_mdb = "~/Desktop/tmb_dev_data_snapshot_100925.json"

# Data from demo M2C2 study on PSU production server
mdb = m2c2.core.pipeline.LASSIE().load(source_name="mongodb", source_path=source_path_mdb)
mdb.assure(required_columns=[
        "study_uid",
        "user_uid",
        "activity_name",
    ])

In [ ]:
mdb.whats_inside()

In [ ]:
mdb.score()
mdb.inspect()

In [ ]:
mdb.summarize(summary_func_map = summary_func_map, groupby_cols=['user_uid'])

In [ ]:
mdb.export(file_basename="MDB", directory="tidy", formats=[".csv", "SQL", "RData"])

## Platform: Qualtrics

In [ ]:
# Define source_folder relative to current working directory
source_folder_qualtrics = os.path.abspath(os.path.join(os.pardir, "datakit/data/qualtrics"))
source_path_qualtrics = f"{source_folder_qualtrics}/M2C2 2 Task_May 2, 2025_10.37.csv"

# Data from Qualtrics
qualtrics = m2c2.core.pipeline.LASSIE().load(source_name="qualtrics", source_path=source_path_qualtrics)
qualtrics.assure(required_columns=['ResponseId'])
qualtrics.score()
qualtrics.summarize(summary_func_map = summary_func_map, groupby_cols=['ResponseId'])
qualtrics.inspect()
qualtrics.export(file_basename="export_qualtrics", directory=output_folder)
qualtrics.export_codebook(filename="codebook_qualtrics.md", directory=output_folder)

# see whats inside
qualtrics.whats_inside()

## Platform: Multiple CSVs of Trial-level data

In [ ]:
# Data from REBOOT Study (UCF and PSU) was manually merged so we have two csvs to load
source_map = {
    "Symbol Search": "~/Documents/GitHub/datakit/data/reboot/m2c2kit_manualmerge_symbol_search_all_ts-20250402_151939.csv",
    "Grid Memory": "~/Documents/GitHub/datakit/data/reboot/m2c2kit_manualmerge_grid_memory_all_ts-20250402_151940.csv"
}

# Data from REBOOT Study (UCF and PSU) was manually merged so we have two csvs to load
mcsv = m2c2.core.pipeline.LASSIE().load(source_name="multicsv", source_map=source_map)
mcsv.assure(required_columns=['participant_id'])
mcsv.score()
mcsv.summarize(summary_func_map = summary_func_map)
mcsv.inspect()
mcsv.export(file_basename="export_multicsv", directory=output_folder)
mcsv.export_codebook(filename="codebook_multicsv.md", directory=output_folder)

# see whats inside
mcsv.whats_inside()

## Platform: M2C2 Production Backend API

This code block might not work. This is an experimental feature right now.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

m2c2api_user = os.getenv("m2c2api_username")
m2c2api_pass = os.getenv("m2c2api_password")

m2c2_api = m2c2.core.pipeline.LASSIE().load(
    source_name="m2c2kit-api",
    source_path="https://api.m2c2kit.com",
    study_id="M2C2_TestNewBackend",
    username=m2c2api_user,
    password=m2c2api_pass,
    payload={"limit": 10},
    pipeline_name="all_uploaded_data",
)
m2c2_api.score()
m2c2_api.summarize(summary_func_map = summary_func_map)
m2c2_api.export(file_basename="export_multicsv", directory=output_folder)